# CreditPFN — Data exploration

Walks through the *sanitised* corpus living under
`data/processed/{pd,lgd}/<dataset_id>.sanitized.csv` plus the
two manifests at `data/manifest_{pd,lgd}.csv`.

**Why these visualisations?** Continued pretraining on a credit-risk
corpus needs to know:

1. how heterogeneous the corpus is in **size** (rows × features),
2. how heterogeneous it is in **target structure** (class imbalance for
   PD, target skew for LGD),
3. how heterogeneous it is in **missingness**,
4. for any single dataset, what its **per-feature distributions** and
   **feature-feature correlations** look like.

These are exactly the four signals TabPFN's in-context learning
prior is exposed to during pretraining, so they tell us where the
synthetic-prior has gaps that real credit data can fill in.

All helpers live in [`src/data/exploration.py`](../src/data/exploration.py)
and return `matplotlib` `Figure` objects.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
REPO = Path(os.getcwd()).resolve()
while not (REPO / 'src' / 'data' / 'exploration.py').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

import pandas as pd
from src.data.exploration import (
    load_manifests, load_sanitized_dataset, corpus_summary_table,
    plot_dataset_size_distribution, plot_missing_rate_distribution,
    plot_class_imbalance_distribution,
    plot_target_distribution_pd, plot_target_distribution_lgd,
    plot_feature_distributions, plot_feature_missingness_heatmap,
    plot_feature_correlation,
)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## 0. Corpus-level overview (both tracks)

One row per dataset, combining the manifest with on-disk shape
checks of the sanitised CSVs.

In [ ]:
summary = corpus_summary_table()
summary

In [ ]:
_ = plot_dataset_size_distribution()  # rows + features per dataset, both tracks

In [ ]:
_ = plot_missing_rate_distribution()  # missing rate per dataset, both tracks

---

# Part 1 — PD (classification)

Probability of Default. Target is binary; the modelling question is
*will this borrower default within the horizon*.

In [ ]:
summary_pd = corpus_summary_table('pd')
summary_pd

## 1.1 Class imbalance

Credit-risk classification is almost always imbalanced — defaults
are rare. The bar chart below shows the share of the *minority*
class in each dataset. A balanced dataset would sit at 0.5; most
credit-risk datasets sit between 0.05 and 0.30.

In [ ]:
_ = plot_class_imbalance_distribution()

## 1.2 Per-dataset target distribution (all PD datasets)

Bar charts of class proportions, one subplot per dataset.

In [ ]:
_ = plot_target_distribution_pd()

## 1.3 Drill-down into individual datasets

Pick one to inspect: feature distributions, missingness pattern,
and feature-feature correlation. Change `dataset_id` to explore
the others.

In [ ]:
dataset_id = '0001.gmsc'   # try also: 0013.hmeq, 0008.german, 0012.home_credit
_ = plot_feature_distributions('pd', dataset_id)

In [ ]:
_ = plot_feature_missingness_heatmap('pd', dataset_id)

In [ ]:
_ = plot_feature_correlation('pd', dataset_id)

---

# Part 2 — LGD (regression)

Loss Given Default. Target is the fraction of exposure lost
given that a default occurred — a real-valued number in `[0, 1]`
by definition. The modelling question is *how much was lost*.

In [ ]:
summary_lgd = corpus_summary_table('lgd')
summary_lgd

## 2.1 Per-dataset target distribution (all LGD datasets)

Histograms of LGD targets. Two structural facts to look for:

* **Mass at 0** — full recovery (often the modal outcome on
  secured-lending datasets).
* **Mass at 1** — total loss.

The interior shape between these two spikes is what TabPFN's
regression head must learn to model — and where the
`_low-skew` / `_quantiles` v2.5 specialist checkpoints are
designed to help (see `papers/LITERATURE.md` § Ye 2025).

In [ ]:
_ = plot_target_distribution_lgd()

## 2.2 Drill-down into individual LGD datasets

In [ ]:
dataset_id = '0001.heloc'  # try also: 0006.lgd_freddie, 0004.base_model
_ = plot_feature_distributions('lgd', dataset_id)

In [ ]:
_ = plot_feature_missingness_heatmap('lgd', dataset_id)

In [ ]:
_ = plot_feature_correlation('lgd', dataset_id)

---

## 3. What to look for next

* Wide-feature datasets (`0011.loan_default`, `0014.algorithmwatch`)
  went through `FeatureAgglomeration`. Their post-sanitise feature
  columns are `feat_agglo_0000`..`0127` — this is expected.
* For LGD, look for the bimodal shape (mass at 0 *and* at 1) on
  the larger datasets (HELOC, lgd_freddie). The interior is
  dominated by ~50–70 % loss rates.
* For PD, look for the consistency of class imbalance — most
  datasets cluster around 10-30 % defaults, with a long tail of
  more imbalanced ones.
* Missingness is highly dataset-specific — some have ≤ 1 % missing
  cells, others have 30 %+.

These corpus characteristics are what TabPFN's continued
pretraining must adapt to. They feed directly into the choice of
training hyperparameters (context size, batch composition,
learning rate) under `src/train/`.